In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time
import random

In [ ]:
import json

with open('app\artifacts\vocab.json', 'r') as file:
    vocab = json.load(file)
len(vocab)

64649

In [3]:
def text_to_indices(sentence, vocab):

  numerical_sentence = []

  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab['<unk>'])

  return numerical_sentence


In [4]:
class LSTMModel(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(input_size=100, hidden_size=200,batch_first=True)
    self.fc = nn.Linear(200, vocab_size)

  def forward(self, x):
    embedded = self.embedding(x)
    intermediate_hidden_states, (final_hidden_state, final_cell_state) = self.lstm(embedded)
    output = self.fc(final_hidden_state.squeeze(0))
    return output

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [6]:
# to load
model = LSTMModel(vocab_size=len(vocab))

model.load_state_dict(
    torch.load("cornell_lstm_next_word.pth", map_location="cpu", weights_only=True)
)

model.eval()

LSTMModel(
  (embedding): Embedding(64649, 100)
  (lstm): LSTM(100, 200, batch_first=True)
  (fc): Linear(in_features=200, out_features=64649, bias=True)
)

In [ ]:
# prediction
from string import punctuation
punc = set(punctuation)
punc.discard("\'")

def prediction(model, vocab, text, top_k):
  model.eval()

  # tokenize
  def process(text):
    #lowercasing, punc, digit, special character removal
    new_text = ""
    for i in text:
        if i not in punc and not i.isdigit():
            new_text+=i
    tokens = new_text.lower().split()
    return tokens

  tokenized_text = process(text)

  # text -> numerical indices
  numerical_text = text_to_indices(tokenized_text, vocab)

  padded_text = torch.tensor(numerical_text, dtype=torch.long).unsqueeze(0).to(device)

  with torch.no_grad():
    # send to model
    output = model(padded_text)


  output = torch.nn.functional.softmax(output, dim = 1)*100
  op = torch.argsort(output, dim = 1, descending=True)[0, :top_k]
  tokens = []
  for i in op:
    tokens.append(list(vocab.keys())[i])

  return tokens, output[0, op]

In [ ]:
prompt = "I need to talk to you for a"
prediction(model, vocab, prompt, 5)

(['while', 'minute', 'moment', 'second', 'little'],
 tensor([26.1715,  8.2149,  6.2974,  5.4582,  4.5298]))

In [ ]:
# generate some tokens using random sampling the next token according to its probability
num_tokens = 50
input_text = "you are here because"

for i in range(num_tokens):
  word, prob = prediction(model, vocab, input_text, 5)
  output_word = random.choices(word, prob.numpy(), k=1)[0]
  input_text = input_text + " " + output_word
  print(input_text)
  time.sleep(0.5)

you are here because i
you are here because i am
you are here because i am sorry
you are here because i am sorry you
you are here because i am sorry you know
you are here because i am sorry you know it
you are here because i am sorry you know it was
you are here because i am sorry you know it was a
you are here because i am sorry you know it was a mistake
you are here because i am sorry you know it was a mistake you
you are here because i am sorry you know it was a mistake you didn't
you are here because i am sorry you know it was a mistake you didn't know
you are here because i am sorry you know it was a mistake you didn't know that
you are here because i am sorry you know it was a mistake you didn't know that you
you are here because i am sorry you know it was a mistake you didn't know that you could
you are here because i am sorry you know it was a mistake you didn't know that you could take
you are here because i am sorry you know it was a mistake you didn't know that you could tak